In [1]:
import pandas as pd
import numpy as np
import igraph as ig
import leidenalg
import math
import networkx as nx
from collections import defaultdict
import os
from scipy.stats import false_discovery_control

In [2]:
#Load and preprocess GRN 
raw_grn = pd.read_csv("/home/quantori/Downloads/data/joung/raw_GRN_for_ehgc.csv")

In [3]:
raw_grn

,Unnamed: 0,source,target,coef_mean,coef_abs,p,-logp
0,0,CREB3L3,A1CF,0.000144,0.000144,2.846976e-05,4.545616
1,1,PRDM1,A1CF,0.000286,0.000286,3.975981e-11,10.400556
2,2,AIRE,A1CF,0.000089,0.000089,1.731293e-06,5.761629
3,3,FOXA2,A3GALT2,0.000330,0.000330,4.504073e-11,10.346395
4,4,FERD3L,A3GALT2,0.000253,0.000253,2.934097e-07,6.532526
...,...,...,...,...,...,...,...
11704,11704,BCL11A,ZNF99,0.000029,0.000029,3.002626e-04,3.522499
11705,11705,MEF2C,ZNF99,-0.000019,0.000019,4.674537e-02,1.330261
11706,11706,LHX1,ZNF99,-0.000065,0.000065,1.396086e-05,4.855088
11707,11707,AIRE,ZNF99,-0.000124,0.000124,3.854296e-10,9.414055


In [4]:
len(set(raw_grn["source"]).union(set(raw_grn["target"])))

1171

In [5]:
#Drop NA or 0 values in P column
raw_grn = raw_grn.dropna(subset=["p"]) 


In [6]:
raw_grn

,Unnamed: 0,source,target,coef_mean,coef_abs,p,-logp
0,0,CREB3L3,A1CF,0.000144,0.000144,2.846976e-05,4.545616
1,1,PRDM1,A1CF,0.000286,0.000286,3.975981e-11,10.400556
2,2,AIRE,A1CF,0.000089,0.000089,1.731293e-06,5.761629
3,3,FOXA2,A3GALT2,0.000330,0.000330,4.504073e-11,10.346395
4,4,FERD3L,A3GALT2,0.000253,0.000253,2.934097e-07,6.532526
...,...,...,...,...,...,...,...
11704,11704,BCL11A,ZNF99,0.000029,0.000029,3.002626e-04,3.522499
11705,11705,MEF2C,ZNF99,-0.000019,0.000019,4.674537e-02,1.330261
11706,11706,LHX1,ZNF99,-0.000065,0.000065,1.396086e-05,4.855088
11707,11707,AIRE,ZNF99,-0.000124,0.000124,3.854296e-10,9.414055


In [7]:
#FDR filtering and only leaving P values less than 0.05

adjusted_pvals = false_discovery_control(raw_grn['p'].values, method='bh')


In [8]:
raw_grn['p'] = adjusted_pvals


In [9]:
raw_grn

,Unnamed: 0,source,target,coef_mean,coef_abs,p,-logp
0,0,CREB3L3,A1CF,0.000144,0.000144,4.236812e-05,4.545616
1,1,PRDM1,A1CF,0.000286,0.000286,1.861446e-10,10.400556
2,2,AIRE,A1CF,0.000089,0.000089,3.001437e-06,5.761629
3,3,FOXA2,A3GALT2,0.000330,0.000330,2.074673e-10,10.346395
4,4,FERD3L,A3GALT2,0.000253,0.000253,5.691740e-07,6.532526
...,...,...,...,...,...,...,...
11704,11704,BCL11A,ZNF99,0.000029,0.000029,4.028157e-04,3.522499
11705,11705,MEF2C,ZNF99,-0.000019,0.000019,5.281180e-02,1.330261
11706,11706,LHX1,ZNF99,-0.000065,0.000065,2.148347e-05,4.855088
11707,11707,AIRE,ZNF99,-0.000124,0.000124,1.361797e-09,9.414055


In [10]:
filtered_grn = raw_grn[raw_grn['p'] <= 0.05]


In [11]:
filtered_grn

,Unnamed: 0,source,target,coef_mean,coef_abs,p,-logp
0,0,CREB3L3,A1CF,0.000144,0.000144,4.236812e-05,4.545616
1,1,PRDM1,A1CF,0.000286,0.000286,1.861446e-10,10.400556
2,2,AIRE,A1CF,0.000089,0.000089,3.001437e-06,5.761629
3,3,FOXA2,A3GALT2,0.000330,0.000330,2.074673e-10,10.346395
4,4,FERD3L,A3GALT2,0.000253,0.000253,5.691740e-07,6.532526
...,...,...,...,...,...,...,...
11703,11703,PAX5,ZNF99,0.000055,0.000055,3.838758e-07,6.714730
11704,11704,BCL11A,ZNF99,0.000029,0.000029,4.028157e-04,3.522499
11706,11706,LHX1,ZNF99,-0.000065,0.000065,2.148347e-05,4.855088
11707,11707,AIRE,ZNF99,-0.000124,0.000124,1.361797e-09,9.414055


In [12]:
len(set(filtered_grn["source"]).union(set(filtered_grn["target"])))

1170

In [ ]:
# Load TF reference
tf_info = pd.read_parquet("hg38_TFinfo_dataframe_gimmemotifsv5_fpr2_threshold_10_20210630.parquet")  ## Google Drive: gCAL_data > Joung2013_results

In [14]:
tf_info

,peak_id,gene_short_name,9430076C15RIK,AC002126.6,AC012531.1,AC226150.2,AFP,AHR,AHRR,AIRE,...,ZNF784,ZNF8,ZNF816,ZNF85,ZSCAN10,ZSCAN16,ZSCAN22,ZSCAN26,ZSCAN31,ZSCAN4
0,chr10_100009853_100010953,DNMBP,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,chr10_100081785_100082885,CPN1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,chr10_100185877_100186977,ERLIN1,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,chr10_100186978_100187057,ERLIN1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,chr10_100229510_100230610,CHUK,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39310,chrY_9721196_9722296,TTTY21B,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
39311,chrY_9735286_9736386,TTTY2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
39312,chrY_9774219_9775319,TTTY1B,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
39313,chrY_9800153_9801253,TTTY22,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [15]:
tf_names = set(tf_info.columns.unique())


In [16]:
tf_names

{'CR925796.2',
 'ZFP28',
 'GRHL2',
 'NR2E3',
 'FOXE3',
 'JDP2',
 'CEBPE',
 'SIX1',
 'TCF3',
 'TCF7',
 'NFKB2',
 'ZFP784',
 'ZNF770',
 'GZF1',
 'SP6',
 'TFEC',
 'ZBTB7B',
 'ZFP957',
 'GFI1B',
 'SREBF1',
 'HOXC5',
 'HESX1',
 'EGR3',
 'RFX8',
 'BCL6B',
 'SP2',
 'E2F5',
 'GATA1',
 'IRF2',
 'KLF13',
 'TBX5',
 'ELF3',
 'OTX2',
 'TFDP1',
 'OBOX1',
 'TEAD2',
 'FOXA',
 'ENSG00000237516',
 'MYF',
 'HOXD1',
 'CDX2',
 'SNAI2',
 'TFE3',
 'FOXJ1',
 'ENO1',
 'ZNF628',
 'NKX3.1',
 'SOX14',
 'DBP',
 'FOXO6',
 'BARX1',
 'ZNF691',
 'MAFK',
 'TBX20',
 'ISL1',
 'ELK4',
 'MYOD1',
 'SOX10',
 'RFX7',
 'ZFP187',
 'TCFEB',
 'AP1',
 'TBP',
 'HOXC13',
 'CMYC',
 'GLI2',
 'ONECUT1',
 'ZNF740',
 'ZSCAN10',
 'FOXP3',
 'SFPI1',
 'ZEB1',
 'ZNF238',
 'E2F2',
 'HOXB4',
 'SOX4',
 'GM239',
 'KLF16',
 'PITX3',
 'ZIC5',
 'BMYB',
 'SMARCA1',
 'TLX3',
 'ZNF816',
 'ZKSCAN1',
 'PAX1',
 'BHLHE41',
 'HOXB9',
 'ETV5',
 'ENSMUSG00000079994',
 'CREB1',
 'FOXO3',
 'HMX2',
 'LHX4',
 'NKX2-5',
 'GBX2',
 'ZFP523',
 'FOXD1',
 'GRE',
 'MEF

In [17]:
tf_names

{'CR925796.2',
 'ZFP28',
 'GRHL2',
 'NR2E3',
 'FOXE3',
 'JDP2',
 'CEBPE',
 'SIX1',
 'TCF3',
 'TCF7',
 'NFKB2',
 'ZFP784',
 'ZNF770',
 'GZF1',
 'SP6',
 'TFEC',
 'ZBTB7B',
 'ZFP957',
 'GFI1B',
 'SREBF1',
 'HOXC5',
 'HESX1',
 'EGR3',
 'RFX8',
 'BCL6B',
 'SP2',
 'E2F5',
 'GATA1',
 'IRF2',
 'KLF13',
 'TBX5',
 'ELF3',
 'OTX2',
 'TFDP1',
 'OBOX1',
 'TEAD2',
 'FOXA',
 'ENSG00000237516',
 'MYF',
 'HOXD1',
 'CDX2',
 'SNAI2',
 'TFE3',
 'FOXJ1',
 'ENO1',
 'ZNF628',
 'NKX3.1',
 'SOX14',
 'DBP',
 'FOXO6',
 'BARX1',
 'ZNF691',
 'MAFK',
 'TBX20',
 'ISL1',
 'ELK4',
 'MYOD1',
 'SOX10',
 'RFX7',
 'ZFP187',
 'TCFEB',
 'AP1',
 'TBP',
 'HOXC13',
 'CMYC',
 'GLI2',
 'ONECUT1',
 'ZNF740',
 'ZSCAN10',
 'FOXP3',
 'SFPI1',
 'ZEB1',
 'ZNF238',
 'E2F2',
 'HOXB4',
 'SOX4',
 'GM239',
 'KLF16',
 'PITX3',
 'ZIC5',
 'BMYB',
 'SMARCA1',
 'TLX3',
 'ZNF816',
 'ZKSCAN1',
 'PAX1',
 'BHLHE41',
 'HOXB9',
 'ETV5',
 'ENSMUSG00000079994',
 'CREB1',
 'FOXO3',
 'HMX2',
 'LHX4',
 'NKX2-5',
 'GBX2',
 'ZFP523',
 'FOXD1',
 'GRE',
 'MEF

In [18]:
len(tf_names)

1096

In [19]:
#Only leaving the TFs that are in base GRN file
filtered_grn = filtered_grn[
    filtered_grn['source'].isin(tf_names) & filtered_grn['target'].isin(tf_names)
]


In [20]:
filtered_grn

,Unnamed: 0,source,target,coef_mean,coef_abs,p,-logp
407,407,FLI1,AIRE,-0.000041,0.000041,4.177123e-04,3.505783
409,409,NR3C1,AIRE,-0.000066,0.000066,1.378691e-04,4.009011
412,412,MEF2C,AIRE,0.000210,0.000210,1.073489e-17,19.000296
413,413,HNF4A,AIRE,0.000028,0.000028,1.485380e-02,1.898947
414,414,MAFK,AIRE,-0.000120,0.000120,1.176911e-09,9.486157
...,...,...,...,...,...,...,...
11695,11695,CREB3L3,ZNF75D,-0.001215,0.001215,3.426221e-06,5.700494
11696,11696,NR3C1,ZNF75D,0.021992,0.021992,2.876696e-09,9.044132
11697,11697,GRHL2,ZNF75D,0.027043,0.027043,1.228759e-12,12.944024
11698,11698,HNF4A,ZNF75D,0.001636,0.001636,7.424884e-06,5.343114


In [21]:
# Sort by absolute coefficient descending
filtered_grn = filtered_grn.sort_values(by='coef_abs', ascending=False).reset_index(drop=True)

In [22]:
filtered_grn

,Unnamed: 0,source,target,coef_mean,coef_abs,p,-logp
0,4002,NPAS3,GLI2,0.100969,0.100969,2.192116e-17,18.637751
1,4142,BCL11A,GRHL2,0.076907,0.076907,6.840731e-18,19.233417
2,979,NR3C1,BCL11A,0.063920,0.063920,5.491807e-18,19.351081
3,5272,BCL11A,KLF8,0.047923,0.047923,7.217016e-16,16.846550
4,8377,NR3C1,PKNOX2,0.045080,0.045080,6.646620e-19,20.400821
...,...,...,...,...,...,...,...
468,948,RUNX1,BARX2,0.000034,0.000034,1.424456e-02,1.917615
469,7362,TFAP2B,NKX3-1,0.000033,0.000033,1.873670e-02,1.795393
470,10518,HNF4A,TFEB,-0.000033,0.000033,2.127717e-02,1.738051
471,8255,NR0B1,PGR,-0.000029,0.000029,1.928049e-02,1.782491


In [23]:
#Create Graph and run Leiden in 10 iterations, each time reducing the last 10% of rows
stepwise_results = []
step_size = math.ceil(len(filtered_grn) * 0.1)

for i in range(10):
    current_df = filtered_grn.iloc[:len(filtered_grn) - i * step_size].copy()
    step_label = f"{100 - i * 10}pct"

    # Build igraph and run Leiden
    edges = list(zip(current_df['source'], current_df['target']))
    weights = current_df['coef_abs'].tolist()
    ig_graph = ig.Graph.TupleList(edges, directed=True)
    ig_graph.es["weight"] = weights
    seed = 50
    partition = leidenalg.find_partition(
        ig_graph,
        leidenalg.RBConfigurationVertexPartition,
        weights=weights,
        seed=seed
    )
    
    # Build NetworkX DiGraph for connected components
    
    G_nx = nx.DiGraph()
    for _, row in current_df.iterrows():
        G_nx.add_edge(row['source'], row['target'], weight=row['coef_abs'])
    G_undirected = G_nx.to_undirected()
    connected_components = list(nx.connected_components(G_undirected))

    
    tf_communities = defaultdict(list)
    for comm_id, community in enumerate(partition):
        for v_index in community:
            tf = ig_graph.vs[v_index]["name"]
            if tf in tf_names:
                tf_communities[comm_id].append(tf)

    tf_community_list = []
    for comm_id, tf_list in tf_communities.items():
        for tf in tf_list:
            tf_community_list.append({"community": comm_id, "tf": tf})

    tf_community_df = pd.DataFrame(tf_community_list)
    tf_comm_path = f"tf_communities_step_{step_label}.csv"
    tf_community_df.to_csv(tf_comm_path, index=False)

    # Finding the edges of a graph which is the TF-TF interaction, assigning names and weight (based on abs_coeff),
    #Lastly checking if TFs are a part of community and if they are of the same community. 
    
    #This is the dictionary created in previous step
    tf_to_community = {tf: comm for comm, tfs in tf_communities.items() for tf in tfs}
    edges_in_community = []

    for edge in ig_graph.es:
        source = ig_graph.vs[edge.source]["name"]
        target = ig_graph.vs[edge.target]["name"]
        weight = edge["weight"]
        if source in tf_to_community and target in tf_to_community:
            if tf_to_community[source] == tf_to_community[target]:
                edges_in_community.append({
                    "community": tf_to_community[source],
                    "source": source,
                    "target": target,
                    "weight": weight
                })

    tf_edge_df = pd.DataFrame(edges_in_community)
    tf_edge_path = f"tf_community_edges_step_{step_label}.csv"
    tf_edge_df.to_csv(tf_edge_path, index=False)

    # Plotting, Partition is a community
    # plot_path = f"/home/giorgi_sokhadze/ml_new_data/plots/plot_step_{step_label}.png"
    # layout = ig_graph.layout("fr")
    # ig.plot(
    #     partition,
    #     layout=layout,
    #     target=plot_path,
    #     vertex_size=10,
    #     bbox=(800, 800),
    #     margin=20,
    #     vertex_label=None
    # )

    # Summary saving
    stepwise_results.append({
        "step": step_label,
        "n_edges": len(current_df),
        "n_leiden_communities": len(partition),
        "tf_counts_per_community": {k: len(v) for k, v in tf_communities.items()},
        "tf_communities_file": tf_comm_path,
        "tf_community_edges_file": tf_edge_path,
        "n_connected_components": len(connected_components),
        # "plot": plot_path
    })



In [24]:
# Final DataFrame
summary_df = pd.DataFrame(stepwise_results)
summary_df.to_csv("tf_tf_grn_summary.csv", index=False)
print("DONE")


DONE
